# Problem 1: Recurrent models from scratch

Run the cells in order with Restart and Run All.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

In [2]:
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

x_example, y_example = make_adding(4, 20, 0)
assert x_example.shape == (4, 20, 2)
assert np.count_nonzero(x_example[:, :, 1] == 1) == 8
print("Adding problem generator ok")


Adding problem generator ok


In [3]:
class MyRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h_prev):
        out = x @ self.W_xh + h_prev @ self.W_hh + self.b_h
        return torch.tanh(out)


class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xi = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hi = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bi = nn.Parameter(torch.zeros(hidden_size))
        self.W_xf = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hf = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bf = nn.Parameter(torch.zeros(hidden_size))
        self.W_xc = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hc = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bc = nn.Parameter(torch.zeros(hidden_size))
        self.W_xo = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_ho = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bo = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h_prev, c_prev):
        i = torch.sigmoid(x @ self.W_xi + h_prev @ self.W_hi + self.bi)
        f = torch.sigmoid(x @ self.W_xf + h_prev @ self.W_hf + self.bf)
        c_hat = torch.tanh(x @ self.W_xc + h_prev @ self.W_hc + self.bc)
        c = f * c_prev + i * c_hat
        o = torch.sigmoid(x @ self.W_xo + h_prev @ self.W_ho + self.bo)
        h = o * torch.tanh(c)
        return h, c


class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xr = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hr = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.br = nn.Parameter(torch.zeros(hidden_size))
        self.W_xz = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hz = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bz = nn.Parameter(torch.zeros(hidden_size))
        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h_prev):
        r = torch.sigmoid(x @ self.W_xr + h_prev @ self.W_hr + self.br)
        z = torch.sigmoid(x @ self.W_xz + h_prev @ self.W_hz + self.bz)
        h_tilde = torch.tanh(x @ self.W_xh + (h_prev @ self.W_hh) * r + self.bh)
        h = (1 - z) * h_tilde + z * h_prev
        return h


In [4]:
def parameter_count_summary():
    rnn = MyRNNCell(9, 64)
    lstm = MyLSTMCell(9, 64)
    gru = MyGRUCell(9, 64)
    return {
        "rnn": sum(p.numel() for p in rnn.parameters()),
        "lstm": sum(p.numel() for p in lstm.parameters()),
        "gru": sum(p.numel() for p in gru.parameters()),
    }

counts = parameter_count_summary()
print(counts)
assert counts["rnn"] == 4736
assert counts["lstm"] == 18944
assert counts["gru"] == 14208


{'rnn': 4736, 'lstm': 18944, 'gru': 14208}


In [5]:
def verify_cells_against_reference():
    torch.manual_seed(0)
    x = torch.randn(8, 9)
    h_prev = torch.randn(8, 64)
    c_prev = torch.randn(8, 64)

    rnn_ref = nn.RNNCell(9, 64, nonlinearity="tanh")
    rnn_ref.weight_ih.data.copy_(torch.randn(64, 9))
    rnn_ref.weight_hh.data.copy_(torch.randn(64, 64))
    rnn_ref.bias_ih.data.zero_()
    rnn_ref.bias_hh.data.zero_()

    lstm_ref = nn.LSTMCell(9, 64)
    lstm_ref.weight_ih.data.copy_(torch.randn(4 * 64, 9))
    lstm_ref.weight_hh.data.copy_(torch.randn(4 * 64, 64))
    lstm_ref.bias_ih.data.zero_()
    lstm_ref.bias_hh.data.zero_()

    gru_ref = nn.GRUCell(9, 64)
    gru_ref.weight_ih.data.copy_(torch.randn(3 * 64, 9))
    gru_ref.weight_hh.data.copy_(torch.randn(3 * 64, 64))
    gru_ref.bias_ih.data.zero_()
    gru_ref.bias_hh.data.zero_()

    rnn_impl = MyRNNCell(9, 64)
    rnn_impl.W_xh.data.copy_(rnn_ref.weight_ih.t())
    rnn_impl.W_hh.data.copy_(rnn_ref.weight_hh.t())
    rnn_impl.b_h.data.copy_(rnn_ref.bias_ih + rnn_ref.bias_hh)

    lstm_impl = MyLSTMCell(9, 64)
    lstm_impl.W_xi.data.copy_(lstm_ref.weight_ih[:64].t())
    lstm_impl.W_hi.data.copy_(lstm_ref.weight_hh[:64].t())
    lstm_impl.bi.data.copy_(lstm_ref.bias_ih[:64] + lstm_ref.bias_hh[:64])
    lstm_impl.W_xf.data.copy_(lstm_ref.weight_ih[64:128].t())
    lstm_impl.W_hf.data.copy_(lstm_ref.weight_hh[64:128].t())
    lstm_impl.bf.data.copy_(lstm_ref.bias_ih[64:128] + lstm_ref.bias_hh[64:128])
    lstm_impl.W_xc.data.copy_(lstm_ref.weight_ih[128:192].t())
    lstm_impl.W_hc.data.copy_(lstm_ref.weight_hh[128:192].t())
    lstm_impl.bc.data.copy_(lstm_ref.bias_ih[128:192] + lstm_ref.bias_hh[128:192])
    lstm_impl.W_xo.data.copy_(lstm_ref.weight_ih[192:256].t())
    lstm_impl.W_ho.data.copy_(lstm_ref.weight_hh[192:256].t())
    lstm_impl.bo.data.copy_(lstm_ref.bias_ih[192:256] + lstm_ref.bias_hh[192:256])

    gru_impl = MyGRUCell(9, 64)
    gru_impl.W_xr.data.copy_(gru_ref.weight_ih[:64].t())
    gru_impl.W_hr.data.copy_(gru_ref.weight_hh[:64].t())
    gru_impl.br.data.copy_(gru_ref.bias_ih[:64] + gru_ref.bias_hh[:64])
    gru_impl.W_xz.data.copy_(gru_ref.weight_ih[64:128].t())
    gru_impl.W_hz.data.copy_(gru_ref.weight_hh[64:128].t())
    gru_impl.bz.data.copy_(gru_ref.bias_ih[64:128] + gru_ref.bias_hh[64:128])
    gru_impl.W_xh.data.copy_(gru_ref.weight_ih[128:192].t())
    gru_impl.W_hh.data.copy_(gru_ref.weight_hh[128:192].t())
    gru_impl.bh.data.copy_(gru_ref.bias_ih[128:192] + gru_ref.bias_hh[128:192])

    rnn_out = rnn_impl(x, h_prev)
    rnn_ref_out = rnn_ref(x, h_prev)
    assert torch.allclose(rnn_out, rnn_ref_out, atol=1e-5, rtol=1e-5)

    lstm_out, c_out = lstm_impl(x, h_prev, c_prev)
    lstm_ref_out, c_ref = lstm_ref(x, (h_prev, c_prev))
    assert torch.allclose(lstm_out, lstm_ref_out, atol=1e-5, rtol=1e-5)
    assert torch.allclose(c_out, c_ref, atol=1e-5, rtol=1e-5)

    gru_out = gru_impl(x, h_prev)
    gru_ref_out = gru_ref(x, h_prev)
    assert torch.allclose(gru_out, gru_ref_out, atol=1e-5, rtol=1e-5)

    return {
        "rnn_max_diff": float((rnn_out - rnn_ref_out).abs().max().item()),
        "lstm_max_diff": float((lstm_out - lstm_ref_out).abs().max().item()),
        "gru_max_diff": float((gru_out - gru_ref_out).abs().max().item()),
    }

print(verify_cells_against_reference())


{'rnn_max_diff': 0.0, 'lstm_max_diff': 0.0, 'gru_max_diff': 2.384185791015625e-07}


In [6]:
def run_problem1_smoke_tests():
    counts = parameter_count_summary()
    verification = verify_cells_against_reference()
    x, y = make_adding(128, 20, 0)
    x = torch.tensor(x)
    y = torch.tensor(y)

    class MiniModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn = nn.GRU(2, 8, batch_first=True)
            self.head = nn.Linear(8, 1)

        def forward(self, seq):
            _, h = self.rnn(seq)
            return self.head(h.squeeze(0)).squeeze(-1)

    model = MiniModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    for _ in range(5):
        pred = model(x[:32])
        loss = loss_fn(pred, y[:32])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return {
        "counts": counts,
        "verification": verification,
        "loss": float(loss.item()),
    }

print(run_problem1_smoke_tests())


{'counts': {'rnn': 4736, 'lstm': 18944, 'gru': 14208}, 'verification': {'rnn_max_diff': 0.0, 'lstm_max_diff': 0.0, 'gru_max_diff': 2.384185791015625e-07}, 'loss': 1.1354689598083496}
